# Rating Attribution Notebook

Traces a single player performance through every step of the rating pipeline,
showing exactly where the final rating comes from. This notebook is purely
interactive — it doesn't write anything to disk. For a saved report, use
`scripts/full_attribution_report.py` (which calls the same `attribution_lib.py`
functions this notebook does, so the numbers always agree).

**Usage:** paste `MATCH_DATA` and `PERFORMANCE` at the top of cell 1, then run all cells.

**Multi-position performances:** steps 1-6 walk through one position
(`INSPECT_POS`, defaulting to `positions_played[0]`) in depth. Step 7 shows how
the service blends *all* listed positions (mirror-pair collapse, cosine-similarity
drag, versatility bonus) using the real `AttributionService` machinery from
`attribution_lib.py`.

In [1]:
# ── INPUTS — paste your dicts here ───────────────────────────────────────────

TEAM_NAME = "Arsenal"

MATCH_DATA = {
            "in_game_date": "2027-01-27T00:00:00",
            "half_length": 10,
            "competition": "UEFA Champions League",
            "home_team_name": "Dinamo Zagreb",
            "away_team_name": "Arsenal",
            "home_score": 1,
            "away_score": 5,
            "home_stats": {
                "possession": 40,
                "ball_recovery": 13,
                "shots": 4,
                "xg": 1.1,
                "passes": 192,
                "tackles": 16,
                "tackles_won": 5,
                "interceptions": 16,
                "saves": 7,
                "fouls_committed": 1,
                "offsides": 0,
                "corners": 0,
                "free_kicks": 1,
                "penalty_kicks": 0,
                "yellow_cards": 0
            },
            "away_stats": {
                "possession": 60,
                "ball_recovery": 9,
                "shots": 17,
                "xg": 7.0,
                "passes": 280,
                "tackles": 41,
                "tackles_won": 17,
                "interceptions": 15,
                "saves": 3,
                "fouls_committed": 1,
                "offsides": 0,
                "corners": 4,
                "free_kicks": 1,
                "penalty_kicks": 0,
                "yellow_cards": 0
            }
        }

PERFORMANCE = {
                "performance_type": "Outfield",
                "positions_played": [
                    "CDM"
                ],
                "goals": 0,
                "assists": 0,
                "shots": 0,
                "shot_accuracy": 0,
                "passes": 12,
                "pass_accuracy": 100,
                "dribbles": 8,
                "dribble_success_rate": 100,
                "tackles": 1,
                "tackle_success_rate": 0,
                "offsides": 0,
                "fouls_committed": 0,
                "possession_won": 1,
                "possession_lost": 0,
                "minutes_played": 21,
                "distance_covered": 2.1,
                "distance_sprinted": 0.7,
                "match_rating": 6.7,
                "player_id": 15
            }

In [2]:
from pathlib import Path
import sys, json, math
import numpy as np

project_root = Path("..").resolve().parent
ratings_creation_dir = Path(".").resolve()
for _p in (str(project_root), str(ratings_creation_dir)):
    if _p not in sys.path:
        sys.path.append(_p)

# AttributionService and all report-building logic live in attribution_lib.py
# (alongside this notebook) so this notebook and scripts/full_attribution_report.py
# — which does the same trace for EVERY position on a multi-position performance —
# never drift apart.
from attribution_lib import (
    AttributionService,
    FBWB_DRIBBLE_THRESHOLD,
    FBWB_POS,
    FBWB_POSS_LOST_FLOOR,
    LOG_STATS,
    MASTERY_NAMES,
    NEG_STATS,
    build_hybrid_report,
    load_config,
    run_attribution,
)

weights, means_stds = load_config(project_root)
print("Library loaded.")

Library loaded.


In [3]:
# Run the service — this populates svc.position_snapshots with one full
# attribution snapshot per position in PERFORMANCE["positions_played"].
svc, final_rating, blend = run_attribution(
    weights=weights,
    means_stds=means_stds,
    match_data=MATCH_DATA,
    performance=PERFORMANCE,
    team_name=TEAM_NAME,
    half_length=MATCH_DATA["half_length"],
)

if PERFORMANCE.get("performance_type") == "GK":
    raise ValueError(
        "This notebook only traces Outfield performances "
        "(GK uses different internal methods, none of which are hooked)."
    )

# Steps 1-6 below walk through ONE position in detail. Change this to inspect
# a different listed position; for a full report covering every position at
# once (plus the hybrid blend), use scripts/full_attribution_report.py.
INSPECT_POS = PERFORMANCE["positions_played"][0]
snapshot = svc.position_snapshots[INSPECT_POS]
pos = INSPECT_POS
pm = snapshot["pre_modifier"]
minutes = pm["minutes_played"]
half_len = MATCH_DATA["half_length"]

# Extract match context for display
is_home   = MATCH_DATA["home_team_name"] == TEAM_NAME
team_stats = MATCH_DATA["home_stats"] if is_home else MATCH_DATA["away_stats"]
opp_stats  = MATCH_DATA["away_stats"] if is_home else MATCH_DATA["home_stats"]
team_xg    = team_stats["xg"]
opp_xg     = opp_stats["xg"]
opp_goals  = (MATCH_DATA["away_score"] if is_home else MATCH_DATA["home_score"])

print(f"Match:    {MATCH_DATA['home_team_name']} vs {MATCH_DATA['away_team_name']} "
      f"({MATCH_DATA['home_score']}-{MATCH_DATA['away_score']})")
print(f"Player:   {PERFORMANCE['player_id']}  positions={PERFORMANCE['positions_played']}  "
      f"minutes={minutes}")
if len(PERFORMANCE["positions_played"]) > 1:
    print(f"Inspecting position: {pos}  (others: "
          f"{[p for p in PERFORMANCE['positions_played'] if p != pos]} — "
          f"see the hybrid-blend cell near the end)")
print(f"Team xG:  {team_xg}   Opponent xG: {opp_xg}   Opponent goals: {opp_goals}")
print(f"Half length: {half_len} min  |  H_BASE: {svc.H_BASE}  |  time scalar: {svc.H_BASE/half_len}")
print(f"Possession: {team_stats['possession']}%")
print()
print(f"SERVICE FINAL RATING: {final_rating}")
print(f"Position rating for {pos} alone: {snapshot['position_final_rating']:.4f}")

Match:    Dinamo Zagreb vs Arsenal (1-5)
Player:   15  positions=['CDM']  minutes=21
Team xG:  7.0   Opponent xG: 1.1   Opponent goals: 1
Half length: 10 min  |  H_BASE: 10.0  |  time scalar: 1.0
Possession: 60%

SERVICE FINAL RATING: 6.7
Position rating for CDM alone: 6.7175


In [4]:
# ── Step 1: Half-length normalisation ────────────────────────────────────────
# The service applies ONE transformation at this stage:
#   volume stats × (H_BASE / half_length)
# Goals, assists, shots are NOT in vol_columns and are passed through raw.
# NO possession adjustment is applied at inference.

# These are the exact vol_columns from the service (lines 783-793)
VOL_COLS = {
    "passes", "dribbles", "tackles", "possession_won", "possession_lost",
    "fouls_committed", "offsides", "distance_covered", "distance_sprinted",
}
RATE_COLS = {"shot_accuracy", "pass_accuracy", "dribble_success_rate", "tackle_success_rate"}
RARE_COLS = {"goals", "assists", "shots"}  # NOT half-length scaled

time_scalar = svc.H_BASE / half_len
norm        = snapshot["normalized_metrics"]

print(f"Half-length scalar: H_BASE({svc.H_BASE}) / half_length({half_len}) = {time_scalar:.4f}")
print(f"(Goals, assists, shots: passed through raw — not half-length scaled)")
print(f"(Rate stats: passed through raw — not scaled)")
print(f"(No possession adjustment at inference)")
print()
print(f"{'Stat':<28} {'Raw':>8} {'Type':>12} {'Normalized':>12} {'Svc value':>10}")
print("-" * 75)

all_stats = sorted(VOL_COLS | RARE_COLS | RATE_COLS)
for stat in all_stats:
    raw = PERFORMANCE.get(stat, 0.0)
    if stat in VOL_COLS:
        stat_type = "vol ×time"
        normalized = raw * time_scalar
    elif stat in RARE_COLS:
        stat_type = "rare (raw)"
        normalized = raw
    else:
        stat_type = "rate (raw)"
        normalized = raw
    svc_val = norm.get(stat, float("nan"))
    match_flag = "✓" if abs(normalized - svc_val) < 0.01 else f"≠svc:{svc_val:.3f}"
    print(f"{stat:<28} {raw:>8.3f} {stat_type:>12} {normalized:>12.3f} {match_flag:>10}")


Half-length scalar: H_BASE(10.0) / half_length(10) = 1.0000
(Goals, assists, shots: passed through raw — not half-length scaled)
(Rate stats: passed through raw — not scaled)
(No possession adjustment at inference)

Stat                              Raw         Type   Normalized  Svc value
---------------------------------------------------------------------------
assists                         0.000   rare (raw)        0.000          ✓
distance_covered                2.100    vol ×time        2.100          ✓
distance_sprinted               0.700    vol ×time        0.700          ✓
dribble_success_rate          100.000   rate (raw)      100.000          ✓
dribbles                        8.000    vol ×time        8.000          ✓
fouls_committed                 0.000    vol ×time        0.000          ✓
goals                           0.000   rare (raw)        0.000          ✓
offsides                        0.000    vol ×time        0.000          ✓
pass_accuracy                 100

In [5]:
# ── Step 2: Bayesian smoothing → per-90 rates ─────────────────────────────────

LOG_PRIOR = {"possession_won", "possession_lost", "fouls_committed", "offsides"}

# snapshot["p90_metrics"] is the complete per-90 profile the service built for
# this position — captured directly from the p90_metrics argument passed into
# _calculate_z_scores, by which point xt_bonus_p90, non_goal_shots_p90, and the
# raw accuracy columns have all already been merged in.
p90 = dict(snapshot["p90_metrics"])
p90.pop("shots_p90", None)  # not a profile stat; remove from display
pos_ms = snapshot["pos_means_stds"]

print(f"{'Stat':<28} {'Smoothed p90':>13} {'Log xform?':>11} {'→ log value':>12}")
print("-" * 70)
for stat, val in sorted(p90.items()):
    log_flag = "log(x+1)" if stat in LOG_STATS else ""
    log_val  = f"{math.log1p(val):.4f}" if stat in LOG_STATS else ""
    print(f"{stat:<28} {val:>13.4f} {log_flag:>11} {log_val:>12}")

print()
print("Bayesian smoothing formula (volume stats):")
print("  smoothed_p90 = (raw_count + league_avg × (d/90)) / (minutes + d) × 90")
print()
print("Selected stat detail:")
for stat_name, raw_key in [("passes_p90","passes"),("tackles_p90","tackles"),
                            ("possession_won_p90","possession_won"),("goals_p90","goals")]:
    d     = svc.DUMMY_WEIGHTS.get(raw_key, svc.DEFAULT_DUMMY)
    stored_mean = pos_ms.get(stat_name, {}).get("mean", 0.0)
    league_avg = math.expm1(stored_mean) if raw_key in LOG_PRIOR else stored_mean
    raw_count  = snapshot["normalized_metrics"].get(raw_key, 0.0)
    smoothed   = (raw_count + league_avg * (d / 90.0)) / (minutes + d) * 90.0
    print(f"  {stat_name:<26} raw={raw_count:.3f}  d={d}  "
          f"league_avg={league_avg:.3f}  → {smoothed:.4f} (svc: {p90.get(stat_name, float('nan')):.4f})")

Stat                          Smoothed p90  Log xform?  → log value
----------------------------------------------------------------------
assists_p90                         0.0000    log(x+1)       0.0000
distance_covered_p90                9.8359                         
distance_sprinted_p90               3.1977                         
dribble_success_rate              100.0000                         
dribbles_p90                       28.0574                         
fouls_committed_p90                 0.1277    log(x+1)       0.1202
goals_p90                           0.0000    log(x+1)       0.0000
non_goal_shots_p90                  0.0000    log(x+1)       0.0000
offsides_p90                        0.0000    log(x+1)       0.0000
pass_accuracy                     100.0000                         
passes_p90                         45.0291                         
possession_lost_p90                 0.9721    log(x+1)       0.6791
possession_won_p90                  4.1234   

In [6]:
# ── Step 3: Z-scores ──────────────────────────────────────────────────────────

z_scores = snapshot["z_scores"]   # pre-floor values
pos_ms   = snapshot["pos_means_stds"]
STAT_COLS = list(svc._PROFILE_COLS)

print(f"{'Stat':<28} {'p90 value':>10} {'→log':>7} {'mean':>8} {'std':>7} "
      f"{'raw_z':>8} {'neg?':>5} {'vol_mask':>9} {'pre_floor_z':>12}")
print("-" * 100)

for col in STAT_COLS:
    z_key = f"{col}_z"
    final_z = z_scores.get(z_key, 0.0)
    p90_val = p90.get(col) if p90.get(col) is not None else snapshot["normalized_metrics"].get(col, 0.0)
    ms = pos_ms.get(col, {})
    mean = ms.get("mean", 0.0)
    std  = ms.get("std",  1.0)
    log_flag = "✓" if col in LOG_STATS else ""
    log_val  = math.log1p(max(p90_val, 0)) if col in LOG_STATS else p90_val
    raw_z    = ((mean - log_val) / std if col in NEG_STATS else (log_val - mean) / std) if std > 0 else 0.0
    neg_flag = "neg" if col in NEG_STATS else ""
    mask_applied = abs(final_z - raw_z) > 0.001
    mask_str = f"{final_z/raw_z:.3f}×" if (mask_applied and abs(raw_z) > 0.001) else ("0 (low vol)" if mask_applied else "")
    print(f"{col:<28} {p90_val:>10.4f} {log_flag:>7} {mean:>8.4f} {std:>7.4f} "
          f"{raw_z:>8.4f} {neg_flag:>5} {mask_str:>9} {final_z:>12.4f}")

# ── Z-score floors ────────────────────────────────────────────────────────────
pre_floor   = snapshot["z_scores"]
post_floor  = snapshot.get("z_scores_at_dot", {})
pos_floors  = dict(svc.Z_SCORE_FLOORS.get(pos, {}))

goals_raw               = PERFORMANCE.get("goals", 0)
non_goal_shots_smoothed = p90.get("non_goal_shots_p90", 0.0)
perf_eff_eligible       = goals_raw >= 1 and non_goal_shots_smoothed == 0.0

# FB/WB dribble-forgiveness floor: for RB/LB/RWB/LWB, a high dribble output
# (dribbles_p90_z > 1.0) floors possession_lost_p90_z at -1.0 so bombing
# forward isn't penalised for the possession losses that come with it.
fbwb_eligible   = pos in FBWB_POS
dribbles_z_pre  = pre_floor.get("dribbles_p90_z", 0.0)
fbwb_gate_open  = fbwb_eligible and dribbles_z_pre > FBWB_DRIBBLE_THRESHOLD

has_anything = pos_floors or perf_eff_eligible or fbwb_eligible
print()
print("── Z-score floors " + "─" * 56)

if not has_anything:
    print(f"  No floors defined for position '{pos}'.")
else:
    HDR  = f"  {'Stat':<32} {'Floor':>6}  {'Pre':>8}  {'Post':>8}  Status"
    LINE = f"  {'─'*70}"

    def floor_row(stat_z, floor_val, pre, post):
        hit    = post > pre + 0.0001
        arrow  = "↑ HIT" if hit else "—"
        delta  = f"  (+{post-pre:.4f})" if hit else ""
        return f"  {stat_z:<32} {floor_val:>6.2f}  {pre:>8.4f}  {post:>8.4f}  {arrow}{delta}"

    if perf_eff_eligible:
        print(f"  Perfect efficiency fix (goals≥1 and non_goal_shots==0):")
        print(HDR); print(LINE)
        z_key = "non_goal_shots_p90_z"
        pre   = pre_floor.get(z_key, 0.0)
        post  = post_floor.get(z_key, pre)
        print(floor_row(z_key, 0.0, pre, post))
        print()

    if pos_floors:
        print(f"  Position floors ({pos}):")
        print(HDR); print(LINE)
        for stat_z, floor_val in sorted(pos_floors.items()):
            pre  = pre_floor.get(stat_z, 0.0)
            post = post_floor.get(stat_z, pre)
            print(floor_row(stat_z, floor_val, pre, post))
        print()

    if fbwb_eligible:
        print(f"  FB/WB dribble-forgiveness gate (pos in {sorted(FBWB_POS)}):")
        print(f"    dribbles_p90_z > {FBWB_DRIBBLE_THRESHOLD:.2f}: {dribbles_z_pre:.4f} → "
              f"{'OPEN' if fbwb_gate_open else 'CLOSED'}")
        if fbwb_gate_open:
            print(HDR); print(LINE)
            z_key = "possession_lost_p90_z"
            pre   = pre_floor.get(z_key, 0.0)
            post  = post_floor.get(z_key, pre)
            print(floor_row(z_key, FBWB_POSS_LOST_FLOOR, pre, post))

Stat                          p90 value    →log     mean     std    raw_z  neg?  vol_mask  pre_floor_z
----------------------------------------------------------------------------------------------------
goals_p90                        0.0000       ✓   0.0180  0.1058  -0.1704                      -0.1704
assists_p90                      0.0000       ✓   0.0696  0.2083  -0.3342                      -0.3342
non_goal_shots_p90               0.0000       ✓   0.3739  0.2682  -1.3942                      -1.3942
shot_accuracy                    0.0000          18.2731 14.0872  -1.2971         -0.000×       0.0000
passes_p90                      45.0291          36.0697  9.8587   0.9088                       0.9088
pass_accuracy                  100.0000          86.9520  7.2042   1.8112                       1.8111
dribbles_p90                    28.0574          23.6975  7.5641   0.5764                       0.5764
dribble_success_rate           100.0000          93.9898 10.8190   0.5555  

In [7]:
# ── Step 4: Dot product contributions → base_rating ──────────────────────────
# Uses post-floor z-scores (snapshot["z_scores_at_dot"]) to match what the
# service actually passed into _calculate_dot_product.

pos_weights     = weights.get(pos, {})
dot             = snapshot["dot_product"]
impact          = pm["impact_scalar"]
z_scores_floored = snapshot.get("z_scores_at_dot", snapshot["z_scores"])

print(f"{'Stat':<28} {'Weight':>9} {'Z-score (floored)':>18} {'Contribution':>14}")
print("-" * 73)

contribs = []
for col in STAT_COLS:
    w = pos_weights.get(col, 0.0)
    z = z_scores_floored.get(f"{col}_z", 0.0)
    c = w * z
    contribs.append((col, w, z, c))

contribs.sort(key=lambda x: abs(x[3]), reverse=True)

BAR_HALF = 18
max_abs  = max(abs(c) for _, _, _, c in contribs) or 1.0

for col, w, z, c in contribs:
    sign = "+" if c >= 0 else "-"
    units = int(abs(c) / max_abs * BAR_HALF)
    # Mark stats where floor was applied
    pre_z = snapshot["z_scores"].get(f"{col}_z", z)
    floored = abs(z - pre_z) > 0.0001
    floor_marker = " F" if floored else "  "
    if c >= 0:
        bar = f"{'':>{BAR_HALF}}|{'█' * units:<{BAR_HALF}}"
    else:
        bar = f"{'█' * units:>{BAR_HALF}}|{'':>{BAR_HALF}}"
    print(f"{col:<28} {w:>9.5f} {z:>18.4f}{floor_marker} {sign}{abs(c):>12.5f}  {bar}")

print("-" * 73)
print(f"{'DOT PRODUCT':<28} {'':>9} {'':>18}   {dot:>14.5f}")
print()
print(f"  F = floor was applied to this z-score")
print()
print(f"Dot product: {dot:.5f}  ({'POSITIVE → impact_scalar NOT applied' if dot >= 0 else 'NEGATIVE → impact_scalar applied'})")
print(f"Impact scalar: √(min({minutes},90)/90) = {impact:.4f}")
adjusted = dot if dot >= 0 else dot * impact
print(f"Adjusted dot:  {adjusted:.5f}")
print()
base_rating = pm["base_rating"]
print(f"BASE RATING = sigmoid({adjusted:.5f}) = {base_rating:.4f}")


Stat                            Weight  Z-score (floored)   Contribution
-------------------------------------------------------------------------
pass_accuracy                  0.13441             1.8111   +     0.24343                    |██████████████████
possession_lost_p90            0.09208             1.1706   +     0.10778                    |███████           
passes_p90                     0.11395             0.9088   +     0.10356                    |███████           
distance_covered_p90           0.03347            -2.2485   -     0.07526               █████|                  
xt_bonus_p90                   0.03382             1.8622   +     0.06298                    |████              
dribbles_p90                   0.03077             0.5764   +     0.01774                    |█                 
distance_sprinted_p90          0.03557            -0.4773   -     0.01697                   █|                  
fouls_committed_p90            0.05437             0.3105   + 

In [8]:
# ── Step 5: Bonus breakdown ───────────────────────────────────────────────────

iso     = pm["isolation_multiplier"]
impact  = pm["impact_scalar"]
goals   = pm["goals"]
assists = pm["assists"]
opponent_goals = pm["opponent_goals"]
opp_xg_val     = pm["opponent_xg"]
pos_key = pm["pos"]
mp      = pm["minutes_played"]

bonuses = []

# Goal bonus
alpha = svc.GOAL_ALPHA.get(pos_key, 0.0)
if goals >= 1:
    t_goals = goals * (goals + 1) / 2
    gb = alpha * t_goals * iso
    bonuses.append(
        (
            "Goal bonus",
            f"α={alpha} × T({int(goals)})={int(t_goals)} × iso={iso:.3f}",
            round(gb, 4),
            True,
        )
    )
else:
    bonuses.append(("Goal bonus", "goals=0 — did not fire", 0.0, False))

# Assist bonus
gamma = svc.ASSIST_GAMMA.get(pos_key, 0.0)
if assists >= 1:
    t_assists = assists * (assists + 1) / 2
    ab = gamma * t_assists * iso
    bonuses.append(
        (
            "Assist bonus",
            f"γ={gamma} × T({int(assists)})={int(t_assists)} × iso={iso:.3f}",
            round(ab, 4),
            True,
        )
    )
else:
    bonuses.append(("Assist bonus", "assists=0 — did not fire", 0.0, False))

# Mastery bonuses
print(f"{'Bonus / Condition':<35} {'Detail':<55} {'Fired':>6} {'Amount':>8}")
print("=" * 110)
for name, detail, amount, fired in bonuses:
    flag = "YES" if fired else "no"
    print(f"{name:<35} {detail:<55} {flag:>6} {amount:>+8.4f}")

print()
print("Mastery conditions:")
for m in snapshot.get("mastery_log", []):
    key = (m["key_a"], m["key_b"])
    name = MASTERY_NAMES.get(key, f"{m['key_a']} + {m['key_b']}")
    fired_str = "YES" if m["fired"] else "no"
    detail = (f"{m['key_a']}={m['val_a']:.3f}  {m['key_b']}={m['val_b']:.3f}  "
              f"min={m['min_z']:.3f}  thresh={m['threshold']}  excess={m['excess']:.3f}")
    print(f"{name[:35]:<35} {detail:<55} {fired_str:>6} {m['bonus']:>+8.4f}")

# CDM Reliable Pivot
# Gate: minutes >= 45 AND pass_acc >= 88 AND raw_passes >= 15.
# raw_passes is the half-length-normalised, pre-Bayesian-smoothing observed
# pass count (snapshot["normalized_metrics"]["passes"]) — NOT a z-score.
print()
if pos_key == "CDM":
    poss_lost  = pm["possession_lost"]
    pass_acc   = pm["pass_accuracy"]
    raw_passes = snapshot["normalized_metrics"].get("passes", 0.0)
    gate = (
        mp >= svc.CDM_PIVOT_MIN_MINUTES
        and pass_acc >= svc.CDM_PIVOT_MIN_PASS_ACC
        and raw_passes >= svc.CDM_PIVOT_MIN_PASSES_RAW
    )
    print(f"CDM Reliable Pivot gate: mins≥{svc.CDM_PIVOT_MIN_MINUTES}({mp}) AND "
          f"pass_acc≥{svc.CDM_PIVOT_MIN_PASS_ACC}({pass_acc}) AND "
          f"passes_raw≥{svc.CDM_PIVOT_MIN_PASSES_RAW}({raw_passes:.2f}) → {'OPEN' if gate else 'CLOSED'}")
    if gate:
        if poss_lost == 0:
            print(f"  Perfect Metronome: poss_lost==0 → +{svc.CDM_PIVOT_PERFECT_METRONOME}")
        elif poss_lost <= 2:
            print(f"  Reliable Shift: poss_lost={poss_lost}≤2 → +{svc.CDM_PIVOT_RELIABLE_SHIFT}")
        else:
            print(f"  poss_lost={poss_lost} > 2 — neither tier fires")

if pos_key in FBWB_POS:
    print()
    print(f"FB/WB dribble-forgiveness gate: dribbles_p90_z > {FBWB_DRIBBLE_THRESHOLD:.2f}: "
          f"{dribbles_z_pre:.4f} → {'OPEN' if fbwb_gate_open else 'CLOSED'}")
    if fbwb_gate_open:
        print(f"  possession_lost_p90_z floored at {FBWB_POSS_LOST_FLOOR:.2f} "
              f"(pre={pre_floor.get('possession_lost_p90_z', 0.0):.4f}, "
              f"post={post_floor.get('possession_lost_p90_z', 0.0):.4f})")

# Clean sheet
print()
cs_ratio = svc.CS_RATIOS.get(pos_key, 0.0)
if opponent_goals == 0 and cs_ratio > 0:
    ramp = min(mp, 60.0) / 60.0
    if opp_xg_val <= 1.0:   tier_val, tier_name = svc.CS_CB_LOW_XG,  f"low xG (≤1.0, xG={opp_xg_val})"
    elif opp_xg_val < 2.0:  tier_val, tier_name = svc.CS_CB_MID_XG,  f"mid xG (1.0-2.0, xG={opp_xg_val})"
    else:                   tier_val, tier_name = svc.CS_CB_HIGH_XG, f"high xG (≥2.0, xG={opp_xg_val})"
    cs_bonus = tier_val * cs_ratio * ramp
    print(f"Clean sheet ({tier_name}): {tier_val} × ratio={cs_ratio} × ramp={ramp:.3f} = +{cs_bonus:.4f}")
elif opponent_goals == 0 and cs_ratio == 0:
    print(f"Clean sheet: opponent scored 0 but {pos_key} has CS ratio=0 — no bonus")
else:
    print(f"Clean sheet: opponent scored {opponent_goals} — no bonus")

print()
total_bonus = snapshot["total_bonus"]
print(f"TOTAL BONUS: {total_bonus:+.4f}")

Bonus / Condition                   Detail                                                   Fired   Amount
Goal bonus                          goals=0 — did not fire                                      no  +0.0000
Assist bonus                        assists=0 — did not fire                                    no  +0.0000

Mastery conditions:
Destroyer / Dominant Stopper / Enfo tackles_p90_z=-0.001  possession_won_p90_z=0.094  min=-0.001  thresh=1.2  excess=0.000     no  +0.0000
Deep-Lying PM / Ball Playing Def /  passes_p90_z=0.909  dribbles_p90_z=0.576  min=0.576  thresh=1.2  excess=0.000     no  +0.0000

CDM Reliable Pivot gate: mins≥45.0(21) AND pass_acc≥88.0(100) AND passes_raw≥15(12.00) → CLOSED

Clean sheet: opponent scored 1.0 — no bonus

TOTAL BONUS: +0.0000


In [9]:
# ── Step 6: Supremacy scalar and final rating ─────────────────────────────────

sup_raw       = snapshot["supremacy_scalar"]
dot_for_sup   = snapshot.get("dot_product", 0.0)
quality_factor= snapshot["quality_factor"]
sup           = snapshot["adjusted_supremacy"]
base          = pm["base_rating"]
bonus         = snapshot["total_bonus"]
raw_final     = snapshot["raw_final_rating"]

print("── Supremacy scalar " + "─" * 45)
print(f"  Raw scalar:      {sup_raw:+.4f}  (team_xg={snapshot['team_xg']}, opp_xg={snapshot['xg_against']})")
print(f"  Dot product:     {dot_for_sup:+.4f}")
print(f"  Quality factor:  max(0, 1 - {dot_for_sup:.3f}/1.5) = {quality_factor:.4f}")
print(f"  Adjusted:        {sup_raw:.4f} × {quality_factor:.4f} = {sup:+.4f}")
print()
print("── Final calculation " + "─" * 45)
print(f"  base_rating              {base:>+10.4f}")
print(f"  total_bonus              {bonus:>+10.4f}")
print(f"  supremacy (adjusted)     {-sup:>+10.4f}")
print(f"  {'─'*36}")
print(f"  raw final                {raw_final:>+10.4f}")
print(f"  clamped (0–10)           {snapshot['position_final_rating']:>10.4f}")
print(f"  ROUNDED                  {round(snapshot['position_final_rating'], 1):>10.1f}")
print()
if len(PERFORMANCE["positions_played"]) == 1:
    match = "✓" if round(snapshot['position_final_rating'], 1) == final_rating else f"≠ service={final_rating}"
    print(f"Service output: {final_rating}  {match}")
else:
    print(f"This is the {pos}-only rating ({snapshot['position_final_rating']:.4f}); "
          f"the service's actual output ({final_rating}) blends all positions — "
          f"see the hybrid-blend cell below.")

── Supremacy scalar ─────────────────────────────────────────────
  Raw scalar:      +0.2500  (team_xg=7.0, opp_xg=1.1)
  Dot product:     +0.4593
  Quality factor:  max(0, 1 - 0.459/1.5) = 0.6938
  Adjusted:        0.2500 × 0.6938 = +0.1735

── Final calculation ─────────────────────────────────────────────
  base_rating                 +6.8909
  total_bonus                 +0.0000
  supremacy (adjusted)        -0.1735
  ────────────────────────────────────
  raw final                   +6.7175
  clamped (0–10)               6.7175
  ROUNDED                         6.7

Service output: 6.7  ✓


In [10]:
# ── Step 7: Multi-position hybrid blend ───────────────────────────────────────
# Steps 1-6 only trace ONE position in depth. If positions_played has more than
# one entry, the service computes a full rating for EACH one (same steps 1-6,
# just repeated) and then blends them: lateral mirror pairs (LB/RB, LWB/RWB,
# LM/RM, LW/RW) collapse to the higher-rated side, then the remaining ratings
# combine via r_max minus a cosine-similarity-weighted drag, plus a versatility
# bonus for being above-average everywhere. `blend` (computed by run_attribution
# above) already has this; print it here rather than re-deriving it.
#
# For the full step-1-6 detail on EVERY position (not just INSPECT_POS), use
# scripts/full_attribution_report.py — it calls the same report builders as
# this notebook, so the output is equivalent, just exhaustive.
print("\n".join(build_hybrid_report(blend)))

MULTI-POSITION HYBRID BLEND

  Positions played: ['CDM']
  Per-position ratings: CDM=6.72
  Positions after collapse: ['CDM']

  Only one position remains after collapse - no blending applied.
  FINAL RATING: 6.7
